In [1]:
# Main_Script.ipynb — Cell 1: environment + paths

import os, sys, torch

# Point to your scratch repo root
REPO_ROOT = "/scratch/mchoud26/d1"
EVAL_DIR  = os.path.join(REPO_ROOT, "eval")
RESULTS_DIR = os.path.join(EVAL_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

os.chdir(EVAL_DIR)
if EVAL_DIR not in sys.path:
    sys.path.append(EVAL_DIR)

print("CWD:", os.getcwd())
print("Files:", os.listdir("."))

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Sun Nov  9 03:16:48 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  |   00000000:01:00.0 Off |                    0 |
| N/A   36C    P0             86W /  500W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2: install deps (run once per new environment)
!pip install --user "transformers>=4.40" "datasets>=2.17" "peft>=0.10" accelerate einops tqdm


Cloning into 'd1'...
remote: Enumerating objects: 216, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 216 (delta 101), reused 75 (delta 75), pack-reused 73 (from 1)
Receiving objects: 100% (216/216), 10.25 MiB | 12.09 MiB/s, done.
Resolving deltas: 100% (119/119), done.
Updating files: 100% (127/127), done.
/scratch/mchoud26/d1


In [ ]:
import os
os.environ["HF_TOKEN"] = <your_token>

# Or rely on hf CLI login you've already done in this account.
print("HF_TOKEN set:", "HF_TOKEN" in os.environ)


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 43.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu124


In [4]:
# Cell 4: shared config

MODEL_BASE = "GSAI-ML/LLaDA-8B-Instruct"

# Your LoRA SFT checkpoint on HF
SFT_LORA  = "sinhasagar507/llada_mix_temp07_ckpt942-1762230332"

DATASET   = "gsm8k"
GEN_LEN   = 256
STEPS     = 64
BLOCK_LEN = 32
MAX_EX    = 200   # bump if you want more examples

print("Base model:", MODEL_BASE)
print("SFT LoRA :", SFT_LORA)

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 95.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 15.2 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 114.0 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.8 MB/s  0:00:00
  Attempting uninstall: fsspec━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/14 [joblib]
    Found existing installation: fsspec 2025.9.0━━━━━━━━━━━━━━  3/14 [joblib]
    Uninstalling fsspec-2025.9.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
      Successfully uninstalled fsspec-2025.9.0━━━━━━━━━━━━━━━━  4/14 [fsspec]
  Attempting uninstall: dill╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/14 [fsspec]
    Found existing installation: dill 0.4.0━━━━━━━━━━━━━━━━━━━  4/

In [5]:
# Cell 5: BASE entropy + flips

base_entropy_out = os.path.join(
    RESULTS_DIR, f"gsm8k_llada_BASE_entropy_flips.json"
)

!python entropy_eval.py \
  --model_path {MODEL_BASE} \
  --dataset {DATASET} \
  --batch_size 1 \
  --gen_length {GEN_LEN} \
  --diffusion_steps {STEPS} \
  --block_length {BLOCK_LEN} \
  --max_examples {MAX_EX} \
  --temperature 0.0 \
  --cfg_scale 0.0 \
  --remasking low_confidence \
  --mask_id 126336 \
  --output_path {base_entropy_out}

print("BASE entropy/flips saved to:", base_entropy_out)


In [6]:
# Cell 6: SFT entropy + flips

sft_entropy_out = os.path.join(
    RESULTS_DIR, f"gsm8k_llada_SFT_entropy_flips.json"
)

!python entropy_eval.py \
  --model_path {MODEL_BASE} \
  --checkpoint_path {SFT_LORA} \
  --dataset {DATASET} \
  --batch_size 1 \
  --gen_length {GEN_LEN} \
  --diffusion_steps {STEPS} \
  --block_length {BLOCK_LEN} \
  --max_examples {MAX_EX} \
  --temperature 0.0 \
  --cfg_scale 0.0 \
  --remasking low_confidence \
  --mask_id 126336 \
  --output_path {sft_entropy_out}

print("SFT entropy/flips saved to:", sft_entropy_out)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_llada.py: 0.00B [00:00, ?B/s]

...

The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:126081 for open-end generation.


You are a helpful math tutor. Solve: 12 * 7 =  










In [ ]:
# Cell 7: BASE FLI

base_fli_out = os.path.join(
    RESULTS_DIR, "gsm8k_llada_BASE_fli.json"
)

!python fli_eval.py \
  --model_path {MODEL_BASE} \
  --dataset {DATASET} \
  --batch_size 1 \
  --gen_length {GEN_LEN} \
  --diffusion_steps {STEPS} \
  --block_length {BLOCK_LEN} \
  --max_examples {MAX_EX} \
  --output_path {base_fli_out}

print("BASE FLI saved to:", base_fli_out)


In [ ]:
# Cell 8: SFT FLI

sft_fli_out = os.path.join(
    RESULTS_DIR, "gsm8k_llada_SFT_fli.json"
)

!python fli_eval.py \
  --model_path {MODEL_BASE} \
  --checkpoint_path {SFT_LORA} \
  --dataset {DATASET} \
  --batch_size 1 \
  --gen_length {GEN_LEN} \
  --diffusion_steps {STEPS} \
  --block_length {BLOCK_LEN} \
  --max_examples {MAX_EX} \
  --output_path {sft_fli_out}

print("SFT FLI saved to:", sft_fli_out)


In [ ]:
# Cell 9: Entropy + flips plots

import json, matplotlib.pyplot as plt

def load_json(path):
    with open(path) as f:
        return json.load(f)

base = load_json(base_entropy_out)
sft  = load_json(sft_entropy_out)

def plot_entropy(d, label):
    if d["mean_entropy_correct"] is None:
        print(label, "has no correct examples.")
        return
    t = range(len(d["mean_entropy_correct"]))
    plt.plot(t, d["mean_entropy_correct"], label=f"{label} correct")
    plt.plot(t, d["mean_entropy_wrong"],   label=f"{label} wrong")

plt.figure(figsize=(6,4))
plot_entropy(base, "BASE")
plot_entropy(sft,  "SFT")
plt.xlabel("Diffusion step")
plt.ylabel("Entropy (generative region)")
plt.title("Entropy-over-steps: BASE vs SFT")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
def plot_flips(d, label):
    if d["mean_flips_correct"] is None:
        print(label, "has no correct examples.")
        return
    t = range(len(d["mean_flips_correct"]))
    plt.plot(t, d["mean_flips_correct"], label=f"{label} correct")
    plt.plot(t, d["mean_flips_wrong"],   label=f"{label} wrong")

plt.figure(figsize=(6,4))
plot_flips(base, "BASE")
plot_flips(sft,  "SFT")
plt.xlabel("Diffusion step")
plt.ylabel("# token flips (generated region)")
plt.title("Token flips per step: BASE vs SFT")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Cell 10: FLI + stability + heatmaps

base_fli = load_json(base_fli_out)
sft_fli  = load_json(sft_fli_out)

steps = range(len(base_fli["mean_fli_correct"]))

# --- FLI ---
plt.figure(figsize=(6,4))
plt.plot(steps, base_fli["mean_fli_correct"], label="BASE correct")
plt.plot(steps, base_fli["mean_fli_wrong"],  "--", label="BASE wrong")
plt.plot(steps, sft_fli["mean_fli_correct"], label="SFT correct")
plt.plot(steps, sft_fli["mean_fli_wrong"],  "--", label="SFT wrong")
plt.xlabel("Diffusion step")
plt.ylabel("FLI (1 - normalized positional entropy)")
plt.title("Flip Localization Index over steps")
plt.legend()
plt.tight_layout()
plt.show()

# --- Stability ---
plt.figure(figsize=(6,4))
plt.plot(steps, base_fli["mean_stability_correct"], label="BASE correct")
plt.plot(steps, base_fli["mean_stability_wrong"],  "--", label="BASE wrong")
plt.plot(steps, sft_fli["mean_stability_correct"], label="SFT correct")
plt.plot(steps, sft_fli["mean_stability_wrong"],  "--", label="SFT wrong")
plt.xlabel("Diffusion step")
plt.ylabel("Fraction of tokens stabilized")
plt.title("Stability of token predictions over steps")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Heatmaps (correct only, BASE vs SFT) ---

import numpy as np

base_hm = np.array(base_fli["mean_posmask_correct"])  # [S, G]
sft_hm  = np.array(sft_fli["mean_posmask_correct"])   # [S, G]

fig, axs = plt.subplots(1, 2, figsize=(10,4), sharey=True)

im0 = axs[0].imshow(base_hm, origin="lower", aspect="auto")
axs[0].set_title("BASE: flip heatmap (correct)")
axs[0].set_xlabel("Token position (gen region)")
axs[0].set_ylabel("Diffusion step")
fig.colorbar(im0, ax=axs[0], label="P(flip)")

im1 = axs[1].imshow(sft_hm, origin="lower", aspect="auto")
axs[1].set_title("SFT: flip heatmap (correct)")
axs[1].set_xlabel("Token position (gen region)")
fig.colorbar(im1, ax=axs[1], label="P(flip)")

plt.tight_layout()
plt.show()
